# Contrôles GPU — occlusion sur population et `predict.py` avec les vrais poids

Deux contrôles demandés par l'audit du 4 novembre, sur **ODIR** (là où le
raccourci d'appareil avait été trouvé) :

1. **Occlusion sur un échantillon tiré au hasard** (15 malades + 15 sains par
   classe, erreurs comprises), sur les **mêmes images** avec la tête `none` et
   la tête `crop-mask` : le raccourci est-il réduit sur la population, et pas
   seulement sur 3 exemples choisis ?
2. **`predict.py` avec les vrais poids RETFound** : redonne-t-il les mêmes
   probabilités que la chaîne d'entraînement ?

Aucune décision n'est prise sur le test : tout se passe sur la validation.

**Avant de lancer (Kaggle)** :
1. `Settings > Accelerator > GPU T4 x2` (pas P100) et `Settings > Internet > On`.
2. `Add-ons > Secrets` → `HF_TOKEN` coché pour ce notebook.
3. Facultatif, pour ne pas tout retélécharger ni recalculer à chaque session
   (`Add Input`) :
   - le dataset **ocular-disease-recognition-odir5k** (andrewmvd) : pas de
     téléchargement des images ;
   - un dataset privé avec les **embeddings** `odir_none_224_RETFound_mae_natureCFP.npz`
     et `odir_crop-mask_224_RETFound_mae_natureCFP.npz` (dossier `embeddings/`
     du dépôt local) : ~20 min d'extraction GPU évitées ;
   - un dataset privé avec **`RETFound_mae_natureCFP.pth`** : pas de
     téléchargement Hugging Face (le secret `HF_TOKEN` devient inutile).
   Le notebook les trouve tout seul dans `/kaggle/input` ; sinon il télécharge
   ou recalcule.
4. `Run All` (~45 min avec les inputs, dont ~40 min d'occlusion ; ~65 min sans).
5. À la fin, télécharger `visual_health_gpu_checks.zip` depuis le panneau `Output`.

Fonctionne aussi sur Colab (GPU T4, secret `HF_TOKEN`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : activer un GPU T4"


In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
![ -d visual-health-platform ] || git clone -q https://github.com/sumarc183-design/visual-health-platform.git
!cd visual-health-platform && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt kagglehub

In [ ]:
import glob
import shutil

INPUTS = "/kaggle/input" if ON_KAGGLE else None
TARGET = "third_party/retfound/checkpoints/RETFound_mae_natureCFP.pth"
os.makedirs(os.path.dirname(TARGET), exist_ok=True)
found = sorted(glob.glob(f"{INPUTS}/**/RETFound_mae_natureCFP.pth", recursive=True)) if INPUTS else []
if os.path.exists(TARGET):
    print("Poids RETFound déjà présents")
elif found:
    os.symlink(found[0], TARGET)  # lien vers l'input Kaggle : rien à copier ni télécharger
    print("Poids RETFound pris dans les inputs :", found[0])
else:
    # Token lu depuis les Secrets, jamais écrit dans le notebook
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    !python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
!ls -lh third_party/retfound/checkpoints

In [ ]:
import glob
import json
import time

import kagglehub

DATA_ROOT = kagglehub.dataset_download("andrewmvd/ocular-disease-recognition-odir5k")
CHECKPOINT = sorted(glob.glob("third_party/retfound/checkpoints/*.pth"))[0]
CACHE_DIR = f"{WORK}/embeddings"
os.makedirs(CACHE_DIR, exist_ok=True)
# Embeddings importés en input : copiés dans le cache, l'extraction GPU est évitée.
for cached in glob.glob("/kaggle/input/**/odir_*_224_RETFound_mae_natureCFP.npz", recursive=True) if ON_KAGGLE else []:
    destination = os.path.join(CACHE_DIR, os.path.basename(cached))
    if not os.path.exists(destination):
        shutil.copy(cached, destination)
        print("Embeddings pris dans les inputs :", os.path.basename(cached))
print(DATA_ROOT, CHECKPOINT)


def check_written(path, since, what):
    """`!cmd | tee` masque les erreurs : on vérifie que la sortie vient d'être écrite."""
    if not os.path.exists(path) or os.path.getmtime(path) < since:
        raise RuntimeError(f"{what} a échoué : {path} absent ou pas mis à jour (voir la sortie ci-dessus).")

## 1. Têtes `none` et `crop-mask` (validation ODIR)

Réentraîne les deux têtes publiées (30 epochs, graine 42) et vérifie qu'on
retrouve les AUROC publiées : 0,840 (`none`) et 0,861 (`crop-mask`). Les
têtes sont enregistrées avec leur manifeste (version du prétraitement,
empreinte du checkpoint).

In [ ]:
REFERENCE = {"none": 0.8404, "crop-mask": 0.8608}
t0 = time.time()
for p in ("none", "crop-mask"):
    !python run_baseline.py --dataset odir --data-root "{DATA_ROOT}" --checkpoint {CHECKPOINT} \
        --preprocess {p} --batch-size 64 --num-workers 2 --embeddings-cache "{CACHE_DIR}" \
        --predictions results/pred_odir_{p}_val.npz --save-head heads/odir_{p}_head.pt \
        --results-json results_odir_{p}.json 2>&1 | grep -v -i warning | tee results_odir_{p}.txt
    check_written(f"results_odir_{p}.json", t0, f"run_baseline.py ({p})")
    auroc = json.load(open(f"results_odir_{p}.json"))["eval"]["auroc_macro"]
    print(f"{p} : AUROC {auroc:.4f} (publiée {REFERENCE[p]:.4f}, écart {auroc - REFERENCE[p]:+.4f})")
    assert abs(auroc - REFERENCE[p]) < 0.002, "résultat différent du chiffre publié : à examiner avant de continuer"

## 2. Occlusion sur un échantillon tiré au hasard

Pour chaque classe : 15 malades et 15 sains tirés au hasard (graine 0), les
mêmes pour les deux têtes. Pour chaque image : part de l'effet d'occlusion
hors de la rétine (à comparer à la part de l'image hors rétine) et variation
du logit quand le fond est remplacé par du gris. La dernière ligne donne la
différence appariée `crop-mask − none` sur les malades, avec son IC à 95 %.

Durée : ~40 min sur T4 en pleine précision (82 000 passages dans RETFound) ; la progression s'affiche classe par classe. `--fp16` divise ce temps par 2 à 3, avec des chiffres légèrement approchés.

In [ ]:
t0 = time.time()
!python -u scripts/occlusion_population.py --dataset odir --data-root "{DATA_ROOT}" --checkpoint {CHECKPOINT} \
    --heads heads/odir_none_head.pt heads/odir_crop-mask_head.pt \
    --positives 15 --negatives 15 --seed 0 --batch-size 64 \
    --out results/occlusion_population_odir.json 2>&1 | grep -v -i warning
check_written("results/occlusion_population_odir.json", t0, "occlusion_population.py")

## 3. `predict.py` avec les vrais poids

Sur 20 images de validation tirées au hasard, `Predictor` (la chaîne de
`predict.py`) doit redonner les probabilités de `run_baseline.py`
(`results/pred_odir_crop-mask_val.npz`), puis la commande est lancée sur 3
images.

In [ ]:
import numpy as np
from src.data.odir_dataset import ODIRDataset
from src.pipeline.inference import Predictor

predictor = Predictor("heads/odir_crop-mask_head.pt", CHECKPOINT, "results/decision_odir.json", device="cuda")
val = ODIRDataset(DATA_ROOT, split="val", preprocess="crop-mask")
reference = np.load("results/pred_odir_crop-mask_val.npz")
row_of = {k: i for i, k in enumerate(reference["keys"])}
picks = np.random.default_rng(0).choice(len(val), 20, replace=False)
diffs = []
for i in picks:
    probs = predictor.probabilities(val.image_path(i)).numpy()
    diffs.append(np.abs(probs - reference["probs"][row_of[val.image_keys[i]]]).max())
check = {"images": len(picks), "max_abs_diff": float(max(diffs)), "warnings": predictor.warnings}
print(check)
json.dump(check, open("results/predict_check_odir.json", "w"), indent=2, ensure_ascii=False)
assert check["max_abs_diff"] < 1e-3, "predict.py ne redonne pas les probabilités de l'entraînement"

images = " ".join(f'"{val.image_path(i)}"' for i in picks[:3])
!python predict.py {images} --head heads/odir_crop-mask_head.pt --checkpoint {CHECKPOINT} \
    --thresholds results/decision_odir.json --policy f1 --out results/predict_examples_odir.json

## 4. Garder le résultat

Zip à télécharger depuis le panneau `Output` (Kaggle) ou téléchargement
direct (Colab) : mesures d'occlusion, contrôle de `predict.py`, résultats et
têtes avec manifeste.

In [ ]:
import zipfile
outputs = [
    "results/occlusion_population_odir.json",
    "results/predict_check_odir.json",
    "results/predict_examples_odir.json",
    "results_odir_none.json",
    "results_odir_crop-mask.json",
    "heads/odir_none_head.pt",
    "heads/odir_crop-mask_head.pt",
]
outputs = [f for f in outputs if os.path.exists(f)]
archive = f"{WORK}/visual_health_gpu_checks.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in outputs:
        z.write(f)
print(len(outputs), "fichiers ->", archive)
if not ON_KAGGLE:
    from google.colab import files
    files.download(archive)